# Problem Set 2: CEO Narrative Sentiment and Stock Returns
## S&P 500 Annual 10-K Filings - SEC EDGAR (2012-2024)

### Research Question
Does the linguistic sentiment of CEO annual narratives in S&P 500 10-K filings predict subsequent stock returns, and does this relationship vary across industries and macroeconomic regimes?

### Motivation and Significance
The Management Discussion and Analysis (MD&A) section of SEC 10-K filings is the legally mandated CEO narrative included in every annual report. Unlike voluntary disclosures, MD&A sections follow a standardized Item 7 structure under SEC Regulation S-K, making them directly comparable across companies and years. Loughran and McDonald (2011) demonstrated that negative word frequency in 10-K filings predicts abnormal returns. This study extends that framework using three sentiment methods including modern transfer learning (FinBERT), alongside unsupervised topic modelling, across a 417-document corpus spanning 68 S&P 500 companies and 11 GICS sectors from 2012 to 2024.

### How to Use This Notebook
This is the main analysis notebook. It loads pre-built data files from the `data/` folder and runs all NLP analysis, visualization, and correlation tests. No internet connection or data scraping is required.

If you want to understand or reproduce how the data files were generated from scratch, see the companion notebook: `00_Data_Generation.ipynb`.

### Data Files Required
Place these files in the `data/` folder before running:
- `mda_corpus.csv`: 417 MD&A text extracts (provided in repository)
- `Loughran-McDonald_MasterDictionary_1993-2024.csv`: download from sraf.nd.edu
- `stock_returns.csv`: annual stock returns (provided in repository)

### Pipeline Overview
| Phase | Content | Method |
|-------|---------|--------|
| 1 | Environment Setup | Library imports and configuration |
| 2 | Corpus Validation | Descriptive statistics and quality checks |
| 3 | Sentiment - VADER and LM | Lexicon-based scoring |
| 4 | Sentiment - FinBERT | Transfer learning scoring |
| 5 | LDA Topic Modelling | Unsupervised with coherence validation |
| 6 | Correlation with Returns | Pearson and Spearman by sector and regime |
| 7 | Synthesis and Limitations | Critical reflection on findings |

# Phase 1: Environment Setup

### Goal
Install and import all required libraries and configure global settings.

### Dependencies
- vaderSentiment: general-purpose rule-based sentiment lexicon

- transformers and torch: FinBERT transfer learning model from HuggingFace

- gensim: LDA topic modelling and coherence validation

- yfinance: historical stock return retrieval (only needed if stock_returns.csv is missing)

- scipy: Pearson and Spearman correlation tests

In [ ]:
# Install all required packages
# Safe to re-run - pip skips already-installed packages
import sys
!{sys.executable} -m pip install -q vaderSentiment transformers torch gensim \
    nltk scipy seaborn matplotlib pandas numpy yfinance tqdm requests beautifulsoup4

In [ ]:
# Core libraries
import os
import re
import time
import warnings
import requests
import numpy as np
import pandas as pd

In [ ]:
# Visualization libraries
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

In [ ]:
# NLP libraries
import nltk
import torch
import torch.nn.functional as F

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

In [ ]:
# Statistical and utility libraries
from pathlib import Path
from scipy import stats
from tqdm import tqdm

warnings.filterwarnings("ignore")

In [ ]:
# Download NLTK tokenisation data
try:
    nltk.download("punkt",     quiet=True)
    nltk.download("punkt_tab", quiet=True)
    nltk.download("stopwords", quiet=True)
    print("NLTK data ready")
except Exception:
    print("NLTK download failed - using cached data if available")

In [ ]:
# Global configuration
RANDOM_STATE = 42

# Crisis periods used for annotation across all plots and regime analysis
CRISIS_REGIMES = {
    "Dotcom Bust": (2000, 2002),
    "GFC":         (2007, 2009),
    "COVID":       (2020, 2020),
    "Inflation":   (2022, 2023),
}

# Directory paths - adjust if your folder structure differs
DATA_DIR   = Path("../data")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Environment ready")
print(f"  PyTorch device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"  Data directory: {DATA_DIR.resolve()}")

# Phase 2: Corpus Validation and Descriptive Analysis

### Goal
Load the pre-built corpus and validate its quality and composition before applying any NLP methods.

### Objectives
- Confirm the corpus contains 417 documents across 68 companies and 11 GICS sectors

- Verify temporal coverage from 2012 to 2024

- Document word count distribution to justify proportional sentiment metrics

- Establish the effective analytical sample size relative to Problem Set 1

### Why 417 Independent Units Matters
Each row in the corpus represents one company-year MD&A document which is a genuinely independent observation with no hidden aggregation. Unlike datasets where many rows share a single analytical unit (such as repeated samples from the same subject), this corpus has a one-to-one correspondence between rows and independent units. The 350 lagged analysis pairs therefore represent 350 truly independent data points, providing sufficient statistical power to detect correlations of r = 0.150 or larger at 80% confidence.

In [ ]:
# Load the corpus
# This file is provided in the data/ folder of the repository
df_corpus = pd.read_csv(DATA_DIR / "mda_corpus.csv")

print("Corpus Summary")
print("=" * 45)
print(f"  Total documents:     {len(df_corpus):,}")
print(f"  Unique companies:    {df_corpus['ticker'].nunique()}")
print(f"  Year range:          {df_corpus['year'].min()}-{df_corpus['year'].max()}")
print(f"  Sectors covered:     {df_corpus['sector'].nunique()}")
print(f"  Mean word count:     {df_corpus['word_count'].mean():,.0f}")
print(f"  Median word count:   {df_corpus['word_count'].median():,.0f}")
print(f"  Min word count:      {df_corpus['word_count'].min():,}")
print(f"  Max word count:      {df_corpus['word_count'].max():,}")

In [ ]:
# Filings per year - verify temporal coverage and crisis period representation
yearly_counts = df_corpus.groupby("year").size().reset_index(name="filings")

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(yearly_counts["year"], yearly_counts["filings"],
       color="#2E75B6", alpha=0.8, width=0.7)

for regime, (y_start, y_end) in CRISIS_REGIMES.items():
    ax.axvspan(y_start - 0.4, y_end + 0.4, alpha=0.12, color="#E74C3C")
    ax.text((y_start + y_end) / 2, ax.get_ylim()[1] * 0.92,
            regime, ha="center", fontsize=8, color="#E74C3C")

ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Number of Filings", fontsize=12)
ax.set_title("Annual 10-K Filing Count by Year (2012-2024)", fontsize=13)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "filings_per_year.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Sector distribution - confirm cross-industry coverage
sector_counts = df_corpus.groupby("sector").size().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
sector_counts.plot(kind="barh", ax=ax, color="#2E75B6", alpha=0.8)
ax.set_xlabel("Number of Filings", fontsize=12)
ax.set_title("Corpus Distribution by GICS Sector", fontsize=13)
ax.grid(True, axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sector_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Word count distribution - justifies use of proportional metrics over raw counts
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df_corpus["word_count"], bins=50, color="#2E75B6", alpha=0.8, edgecolor="white")
ax.axvline(df_corpus["word_count"].median(), color="#E74C3C", linestyle="--",
           linewidth=1.5, label=f"Median: {df_corpus['word_count'].median():,.0f} words")
ax.set_xlabel("Word Count per MD&A", fontsize=12)
ax.set_ylabel("Number of Documents", fontsize=12)
ax.set_title("MD&A Word Count Distribution", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "word_count_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

### Phase 2 Conclusion: Corpus Validation

The corpus meets all quality requirements for robust NLP analysis:

- Scale: 417 documents across 68 companies and 11 GICS sectors from 2012 to 2024. Each document is one genuinely independent analytical unit (one company, one year, one MD&A filing). Unlike datasets where many rows share a single subject, here every row is a distinct observation. The 417 documents therefore represent 417 true analytical units with no hidden collapse.

- Statistical Power: With 350 lagged analysis pairs, the minimum detectable correlation at 80% power is approximately r = 0.150. Published benchmarks from Loughran and McDonald (2011) report effect sizes of r = 0.10-0.15 for sentiment-return relationships, placing this corpus at the lower threshold of adequate power for detecting effects of that magnitude.

- Sector Coverage: All 11 GICS sectors are represented, enabling cross-industry analysis. Technology (approximately 90 filings), Consumer Discretionary (approximately 75 filings), and Healthcare (approximately 70 filings) are most heavily represented.

- Word Count Distribution: Right-skewed distribution with median 3,080 words and maximum 46,668 words. All downstream sentiment metrics use word-proportion ratios rather than raw counts to control for this variation.

# Phase 3: Sentiment Analysis - VADER and Loughran-McDonald

### Goal
Score each MD&A document using two lexicon-based approaches and compare their outputs to establish a baseline before the transfer learning approach in Phase 4.

### Objectives
- VADER Scoring: Apply a rule-based general sentiment lexicon, computing document-level compound scores averaged across sentences

- LM Scoring: Apply the Loughran-McDonald finance-domain dictionary to compute positive, negative, and uncertainty word proportions

- Baseline Comparison: Quantify where the two lexicons diverge to motivate the FinBERT analysis

### Methodology
VADER averages sentence-level compound scores (-1 to +1) across the full document. The LM Master Dictionary (1993-2024 edition) was constructed from SEC 10-K filings, making it the most domain-appropriate lexicon for this corpus type. Loughran and McDonald (2011) showed that up to 73% of words classified as negative in general lexicons are not negative in financial contexts. The LM dictionary corrects for this systematic misclassification.

In [ ]:
# Initialise VADER sentiment analyser
analyzer = SentimentIntensityAnalyzer()

def score_vader(text):
    """
    Score a document using VADER sentence by sentence.
    Caps input at 50,000 characters for consistent runtime across long documents.
    Returns compound score (-1 to +1) and positive, negative, neutral ratios.
    """
    sentences = sent_tokenize(text[:50000])
    if not sentences:
        return {"vader_compound": 0, "vader_pos": 0, "vader_neg": 0, "vader_neu": 1}
    scores = [analyzer.polarity_scores(s) for s in sentences]
    return {
        "vader_compound": sum(s["compound"] for s in scores) / len(scores),
        "vader_pos":      sum(s["pos"]      for s in scores) / len(scores),
        "vader_neg":      sum(s["neg"]      for s in scores) / len(scores),
        "vader_neu":      sum(s["neu"]      for s in scores) / len(scores),
    }

print("VADER scorer ready")

In [ ]:
# Load the Loughran-McDonald Master Dictionary (provided in the repository)

LM_DICT_PATH = DATA_DIR / "Loughran-McDonald_MasterDictionary_1993-2024.csv"

def load_lm_dictionary(path):
    """
    Load LM word sets for positive, negative, and uncertainty categories.
    Non-zero values in each column indicate membership in that category.
    """
    lm = pd.read_csv(path)
    lm["Word"] = lm["Word"].str.lower()
    return (
        set(lm[lm["Positive"]    != 0]["Word"]),
        set(lm[lm["Negative"]    != 0]["Word"]),
        set(lm[lm["Uncertainty"] != 0]["Word"]),
    )

def score_lm(text, pos_words, neg_words, unc_words):
    """
    Score text using LM finance lexicon word proportions.
    Net sentiment = positive word proportion minus negative word proportion.
    Using proportions rather than counts controls for document length variation.
    """
    words = re.findall(r"\b[a-z]+\b", text.lower())
    total = len(words)
    if total == 0:
        return {"lm_pos_ratio": 0, "lm_neg_ratio": 0,
                "lm_uncertain_ratio": 0, "lm_net_sentiment": 0}
    pos = sum(1 for w in words if w in pos_words) / total
    neg = sum(1 for w in words if w in neg_words) / total
    unc = sum(1 for w in words if w in unc_words) / total
    return {
        "lm_pos_ratio":       pos,
        "lm_neg_ratio":       neg,
        "lm_uncertain_ratio": unc,
        "lm_net_sentiment":   pos - neg,
    }

lm_available = LM_DICT_PATH.exists()
if lm_available:
    pos_words, neg_words, unc_words = load_lm_dictionary(LM_DICT_PATH)
    print(f"LM dictionary loaded:")
    print(f"  Positive words:    {len(pos_words):,}")
    print(f"  Negative words:    {len(neg_words):,}")
    print(f"  Uncertainty words: {len(unc_words):,}")
else:
    print(f"LM dictionary not found at {LM_DICT_PATH}")
    print("Download from: https://sraf.nd.edu/loughranmcdonald-master-dictionary/")
    print("VADER and FinBERT will still run without LM scores")

In [ ]:
# Score all documents with VADER and LM
# Cached after first run - delete sentiment_vader_lm.csv to re-score
# Runtime: approximately 25-30 seconds

vader_lm_path = DATA_DIR / "sentiment_vader_lm.csv"

if vader_lm_path.exists():
    print("Loading VADER/LM scores from cache.")
    df_sent = pd.read_csv(vader_lm_path)
else:
    print(f"Scoring {len(df_corpus):,} documents with VADER and LM...")
    records = []
    for _, row in tqdm(df_corpus.iterrows(), total=len(df_corpus), desc="Scoring"):
        record = {"ticker": row["ticker"], "sector": row["sector"], "year": row["year"]}
        record.update(score_vader(row["text"]))
        if lm_available:
            record.update(score_lm(row["text"], pos_words, neg_words, unc_words))
        records.append(record)
    df_sent = pd.DataFrame(records)
    df_sent.to_csv(vader_lm_path, index=False)
    print(f"Saved: {vader_lm_path}")

# Show available columns dynamically
display_cols = ["ticker", "year", "vader_compound"]
if "lm_net_sentiment" in df_sent.columns:
    display_cols.append("lm_net_sentiment")
print(f"Sentiment scores: {len(df_sent):,} documents")
print(df_sent[display_cols].head(10).to_string(index=False))
print(f"Columns: {df_sent.columns.tolist()}")

In [ ]:
# Aggregate by year and visualize sentiment trends
agg_dict = {"vader_mean": ("vader_compound", "mean")}
if "lm_net_sentiment" in df_sent.columns:
    agg_dict["lm_net_mean"] = ("lm_net_sentiment", "mean")
    agg_dict["lm_neg_mean"] = ("lm_neg_ratio",      "mean")

yearly_sent = df_sent.groupby("year").agg(**agg_dict).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# VADER trend
ax = axes[0]
ax.plot(yearly_sent["year"], yearly_sent["vader_mean"],
        color="#2E75B6", linewidth=2, marker="o", markersize=4)
ax.axhline(0, color="gray", linestyle="--", alpha=0.4)
ax.fill_between(yearly_sent["year"], yearly_sent["vader_mean"], 0,
                where=yearly_sent["vader_mean"] >= 0, alpha=0.15, color="#2E75B6")
ax.fill_between(yearly_sent["year"], yearly_sent["vader_mean"], 0,
                where=yearly_sent["vader_mean"] < 0, alpha=0.2, color="#E74C3C")
ax.set_ylabel("Mean VADER Compound", fontsize=10)
ax.set_title("Corpus Sentiment Over Time: VADER and LM (2012-2024)", fontsize=13)
ax.grid(True, alpha=0.25)

# LM trend (if available)
ax2 = axes[1]
if "lm_net_mean" in yearly_sent.columns:
    ax2.plot(yearly_sent["year"], yearly_sent["lm_net_mean"],
             color="#27AE60", linewidth=2, marker="o", markersize=4, label="LM Net")
    ax2.plot(yearly_sent["year"], yearly_sent["lm_neg_mean"],
             color="#E74C3C", linewidth=1.5, linestyle="--", marker="s",
             markersize=3, alpha=0.7, label="LM Negative Ratio")
    ax2.set_ylabel("LM Sentiment", fontsize=10)
    ax2.legend(fontsize=9)
else:
    ax2.text(0.5, 0.5, "LM dictionary not loaded",
             ha="center", va="center", transform=ax2.transAxes, color="gray")
    ax2.set_ylabel("LM Sentiment (unavailable)", fontsize=10)

ax2.grid(True, alpha=0.25)

for regime, (y_start, y_end) in CRISIS_REGIMES.items():
    for ax in axes:
        ax.axvspan(y_start - 0.4, y_end + 0.4, alpha=0.08, color="#E74C3C")

axes[1].set_xlabel("Year", fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "vader_lm_trends.png", dpi=150, bbox_inches="tight")
plt.show()

### Phase 3 Conclusion: VADER and Loughran-McDonald Sentiment

Lexicon-based sentiment analysis across 417 MD&A documents reveals:

- VADER Trend: Mean compound scores remain positive throughout 2012-2024, consistent with the optimistic register of corporate annual reporting. A dip is visible around COVID-2020 where mean compound scores decline before recovering in 2021-2022, consistent with management acknowledging operational disruption.

- LM Dictionary: Successfully loaded with 354 positive words, 2,355 negative words, and 297 uncertainty words from the 1993-2024 edition. LM net sentiment is consistently negative across the corpus (negative word proportion exceeds positive), reflecting the formally cautious and risk-disclosing language of regulatory filings.

- Domain Gap: LM negative ratios are substantially higher than VADER's negative component for the same documents. This replicates at scale the core finding of Loughran and McDonald (2011) - general lexicons systematically misclassify financial language by failing to recognise finance-specific negative terms like "impairment", "write-down", and "adverse".

- Limitation: Both VADER and LM treat words independently and ignore sentence context. Negation ("not profitable") and hedging ("may decline") are not handled correctly. FinBERT in Phase 4 addresses this using contextual embeddings.

# Phase 4: Sentiment Analysis - FinBERT (Transfer Learning)

### Goal
Apply FinBERT, a BERT model fine-tuned on financial text, to score MD&A sentiment with contextual understanding, and compare its outputs against VADER to quantify the domain-specificity gap.

### Objectives
- FinBERT Scoring: Generate positive, negative, and neutral probability scores per document by averaging sentence-level probabilities

- Model Comparison: Compute Pearson correlation between VADER and FinBERT net scores

- Method Selection: Justify FinBERT as the primary metric for downstream correlation analysis

### Methodology
FinBERT (Araci, 2019) uses contextual embeddings from BERT architecture, pre-trained on financial text and fine-tuned for three-class sentiment classification. Each word's representation is shaped by surrounding context, enabling correct handling of negation and hedging that rule-based lexicons miss. Documents are scored by averaging softmax probabilities across up to 100 sentences.

Note on runtime: FinBERT scoring takes approximately 1-2 hours on CPU and 15-20 minutes on GPU. Results are cached in sentiment_finbert.csv after the first run.

In [ ]:
# Load FinBERT from HuggingFace
# Model: ProsusAI/finbert - BERT fine-tuned on financial communication text
# First run downloads approximately 440MB of model weights to local cache
# All subsequent runs load from the local cache in under 30 seconds

FINBERT_MODEL = "ProsusAI/finbert"
MAX_SENTENCES = 100

print(f"Loading FinBERT: {FINBERT_MODEL}")
fb_tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL)
fb_model     = AutoModelForSequenceClassification.from_pretrained(FINBERT_MODEL)
fb_model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
fb_model.to(device)
print(f"FinBERT loaded on {device}")
print(f"Max sentences scored per document: {MAX_SENTENCES}")

In [ ]:
# FinBERT document scoring function
def score_finbert(text, max_sentences=MAX_SENTENCES):
    """
    Score a document with FinBERT by averaging sentence-level softmax probabilities.
    FinBERT label order: positive=0, negative=1, neutral=2.
    Caps at max_sentences to maintain consistent runtime at corpus scale.
    Net score = positive probability minus negative probability.
    """
    sentences = sent_tokenize(text[:50000])[:max_sentences]
    if not sentences:
        return {"finbert_positive": 0, "finbert_negative": 0,
                "finbert_neutral": 1,  "finbert_net": 0}
    all_probs = []
    for sentence in sentences:
        inputs = fb_tokenizer(sentence, return_tensors="pt",
                              truncation=True, max_length=512, padding=True).to(device)
        with torch.no_grad():
            probs = F.softmax(fb_model(**inputs).logits, dim=-1).squeeze().cpu().tolist()
            all_probs.append(probs)
    return {
        "finbert_positive": sum(p[0] for p in all_probs) / len(all_probs),
        "finbert_negative": sum(p[1] for p in all_probs) / len(all_probs),
        "finbert_neutral":  sum(p[2] for p in all_probs) / len(all_probs),
        "finbert_net":      sum(p[0] for p in all_probs) / len(all_probs) -
                            sum(p[1] for p in all_probs) / len(all_probs),
    }

print("FinBERT scorer ready")

In [ ]:
# Score all documents with FinBERT
# Cached after first run - delete sentiment_finbert.csv to re-score
# Runtime: approximately 1-2 hours on CPU, 15-20 minutes on GPU

finbert_path = DATA_DIR / "sentiment_finbert.csv"

if finbert_path.exists():
    print("Loading FinBERT scores from cache.")
    df_finbert = pd.read_csv(finbert_path)
else:
    print(f"Scoring {len(df_corpus):,} documents with FinBERT...")
    print(f"Device: {device} | Estimated time: {'15-20 min' if device=='cuda' else '1-2 hours'}")
    records = []
    for _, row in tqdm(df_corpus.iterrows(), total=len(df_corpus), desc="FinBERT"):
        scores = score_finbert(row["text"])
        scores.update({"ticker": row["ticker"], "sector": row["sector"], "year": row["year"]})
        records.append(scores)
    df_finbert = pd.DataFrame(records)
    df_finbert.to_csv(finbert_path, index=False)
    print(f"Saved: {finbert_path}")

print(f"FinBERT scores: {len(df_finbert):,} documents")
print(df_finbert[["ticker", "year", "finbert_net",
                  "finbert_positive", "finbert_negative"]].head(10).to_string(index=False))

In [ ]:
# Merge all sentiment scores into one master DataFrame
df_all_sent = df_sent.merge(
    df_finbert[["ticker", "year", "finbert_net", "finbert_positive", "finbert_negative"]],
    on=["ticker", "year"], how="left"
)

display_cols = ["ticker", "year", "vader_compound", "finbert_net"]
if "lm_net_sentiment" in df_all_sent.columns:
    display_cols.insert(3, "lm_net_sentiment")

print(f"Master sentiment dataset: {len(df_all_sent):,} documents")
print(df_all_sent[display_cols].head(10).to_string(index=False))

In [ ]:
# Measure VADER vs FinBERT agreement with Pearson correlation
valid = df_all_sent[["vader_compound", "finbert_net"]].dropna()
r, p  = stats.pearsonr(valid["vader_compound"], valid["finbert_net"])

print(f"VADER vs FinBERT Agreement (n={len(valid):,})")
print(f"  Pearson r = {r:.3f}  |  p = {p:.4e}")
print()
if abs(r) > 0.7:
    print("  Strong agreement - both models capture similar sentiment signal")
elif abs(r) > 0.4:
    print("  Moderate agreement - FinBERT captures domain nuances VADER misses")
else:
    print("  Weak agreement - substantial domain-specific divergence")

In [ ]:
# Visualize VADER vs FinBERT yearly trends
yearly_fb = df_all_sent.groupby("year").agg(
    vader_mean   = ("vader_compound", "mean"),
    finbert_mean = ("finbert_net",    "mean"),
).reset_index()

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(yearly_fb["year"], yearly_fb["vader_mean"],
        color="#2E75B6", linewidth=2, marker="o", markersize=4, label="VADER Compound")
ax.plot(yearly_fb["year"], yearly_fb["finbert_mean"],
        color="#27AE60", linewidth=2, marker="s", markersize=4, label="FinBERT Net")
ax.axhline(0, color="gray", linestyle="--", alpha=0.4)

for regime, (y_start, y_end) in CRISIS_REGIMES.items():
    ax.axvspan(y_start - 0.4, y_end + 0.4, alpha=0.08, color="#E74C3C")
    ax.text((y_start + y_end) / 2, ax.get_ylim()[1] * 0.9,
            regime, ha="center", fontsize=8, color="#E74C3C")

ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Mean Sentiment Score", fontsize=12)
ax.set_title("VADER vs FinBERT Corpus Sentiment by Year (2012-2024)", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "vader_vs_finbert.png", dpi=150, bbox_inches="tight")
plt.show()

### Phase 4 Conclusion: FinBERT Transfer Learning

FinBERT was applied across all 417 MD&A documents and compared against VADER:

- Systematic Divergence: FinBERT assigns lower positivity scores than VADER across the majority of documents. FinBERT mean net scores cluster close to zero or slightly negative while VADER means remain consistently positive. This reflects FinBERT's finance-domain training where terms such as "risk", "impairment", "uncertainty", and "adverse" carry greater negative weight than in general language.

- Model Agreement: Pearson r = 0.481 between VADER compound and FinBERT net scores (p = 1.56e-25, n = 417). Moderate agreement confirms that both models detect the same broad sentiment direction but diverge on magnitude. This divergence quantifies the domain-specificity gap that Loughran and McDonald (2011) identified lexically but that transformer models capture contextually.

- Crisis Sensitivity: FinBERT shows a more pronounced shift toward negative sentiment during COVID-2020 compared to VADER. This suggests it better captures the formally cautious language of crisis-period MD&A disclosures where executives acknowledge uncertainty using phrasing that general lexicons score as neutral.

- Method Selection: FinBERT is selected as the primary metric for correlation analysis in Phase 6. Its domain alignment with financial disclosure text (Araci, 2019) and contextual handling of negation and hedging make it the most appropriate choice.

# Phase 5: LDA Topic Modelling with Coherence Validation

### Goal
Discover the latent thematic structure of the MD&A corpus using Latent Dirichlet Allocation and track how topic prevalence shifts across industries and macroeconomic regimes.

### Objectives
- Coherence Validation: Objectively select the optimal number of topics using C_v coherence across k = 5 to 20

- Topic Interpretation: Label discovered topics by top words and connect to known MD&A content domains

- Temporal Analysis: Plot topic prevalence by year to reveal thematic shifts around crisis periods

- Sector Variation: Compare topic compositions across GICS sectors

### Methodology
LDA (Blei et al., 2003) represents each document as a mixture of topics and each topic as a distribution over words. C_v coherence measures semantic similarity of top words within each topic using a sliding window over the corpus. Higher coherence indicates more interpretable, semantically consistent topics. The k that maximises C_v is selected for the final model without any arbitrary pre-specification.

Note on runtime: Coherence validation across k = 5 to 20 takes approximately 45-60 minutes. Results are not cached - re-running will produce the same optimal k due to the fixed random seed.

In [ ]:
# Extended stopword list for MD&A boilerplate terms
# These appear in virtually every filing and carry no thematic signal
MADA_STOPWORDS = {
    "company", "year", "fiscal", "period", "quarter", "annual",
    "results", "operations", "financial", "management", "discussion",
    "analysis", "pursuant", "item", "form", "report", "sec",
    "include", "including", "also", "may", "would", "could",
    "million", "billion", "thousand", "percent", "increase", "decrease",
}

def preprocess_for_lda(text):
    """
    Prepare text for LDA by tokenising and removing noise.
    Keeps only alphabetic tokens of 3 or more characters.
    Removes English stopwords and MD&A boilerplate.
    """
    stop = set(stopwords.words("english")) | MADA_STOPWORDS
    tokens = word_tokenize(text.lower())
    return [t for t in tokens if t.isalpha() and t not in stop and len(t) >= 3]

print("Preprocessing corpus for LDA...")
lda_texts = [preprocess_for_lda(t) for t in tqdm(df_corpus["text"], desc="Tokenising")]

# Build Gensim dictionary and filter extremes
dictionary = corpora.Dictionary(lda_texts)
dictionary.filter_extremes(no_below=3, no_above=0.9)
corpus_bow  = [dictionary.doc2bow(t) for t in lda_texts]

print(f"Preprocessing complete")
print(f"  Vocabulary size: {len(dictionary):,} terms")
print(f"  Documents:       {len(corpus_bow)}")

df_lda_sample = df_corpus.copy()

In [ ]:
# Coherence validation: fit LDA for k = 5 to 20 and score each
# Fixed seed (RANDOM_STATE=42) ensures identical results on every run
# Runtime: approximately 45-60 minutes

print("Running coherence validation (k = 5-20)...")
print("Estimated time: 45-60 minutes")
print()

N_TOPICS_RANGE  = range(5, 21)
coherence_scores = []

for n in N_TOPICS_RANGE:
    model = LdaModel(
        corpus=corpus_bow, id2word=dictionary,
        num_topics=n, random_state=RANDOM_STATE,
        passes=10, alpha="auto", eta="auto",
    )
    cm = CoherenceModel(model=model, texts=lda_texts,
                        dictionary=dictionary, coherence="c_v")
    score = cm.get_coherence()
    coherence_scores.append((n, score))
    print(f"  k={n:2d} | C_v = {score:.4f}")

optimal_n = max(coherence_scores, key=lambda x: x[1])[0]
print(f"Optimal k = {optimal_n}")

In [ ]:
# Plot coherence curve to visualise the validation result
ns, cs = zip(*coherence_scores)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ns, cs, marker="o", linewidth=2, color="#2E75B6")
ax.axvline(optimal_n, color="#E74C3C", linestyle="--", alpha=0.7,
           label=f"Optimal: k={optimal_n} (C_v={max(cs):.4f})")
ax.set_xlabel("Number of Topics (k)", fontsize=12)
ax.set_ylabel("C_v Coherence Score", fontsize=12)
ax.set_title("LDA Coherence Validation: Selecting Optimal Topic Count", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "coherence_plot.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Fit the final LDA model with optimal k using 20 passes for stability
print(f"Fitting final LDA model: k={optimal_n}, 20 passes...")

lda_final = LdaModel(
    corpus=corpus_bow, id2word=dictionary,
    num_topics=optimal_n, random_state=RANDOM_STATE,
    passes=20, alpha="auto", eta="auto",
)

print(f"Top words per topic (k={optimal_n}):")
print("=" * 65)
topic_labels = {}
for i in range(optimal_n):
    top_words = [w for w, _ in lda_final.show_topic(i, topn=10)]
    label = " / ".join(top_words[:3])
    topic_labels[i] = label
    print(f"  Topic {i:2d}: {', '.join(top_words)}")
print("=" * 65)

In [ ]:
# Extract per-document topic distributions and save
topic_records = []
for idx, (_, row) in enumerate(df_lda_sample.iterrows()):
    bow  = corpus_bow[idx]
    dist = dict(lda_final.get_document_topics(bow, minimum_probability=0))
    rec  = {"ticker": row["ticker"], "sector": row["sector"], "year": row["year"]}
    for t in range(optimal_n):
        rec[f"topic_{t}"] = dist.get(t, 0.0)
    topic_records.append(rec)

df_topics   = pd.DataFrame(topic_records)
topic_cols  = [c for c in df_topics.columns if c.startswith("topic_")]
df_topics["dominant_topic"] = df_topics[topic_cols].idxmax(axis=1)
df_topics.to_csv(DATA_DIR / "topic_distributions.csv", index=False)
print(f"Topic distributions saved: {len(df_topics):,} documents")

In [ ]:
# Topic prevalence over time - stacked area chart
topic_by_year = df_topics.groupby("year")[topic_cols].mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
colors = sns.color_palette("tab20", len(topic_cols))
ax.stackplot(
    topic_by_year["year"],
    [topic_by_year[col] for col in topic_cols],
    labels=[f"T{i}: {topic_labels.get(i,'')}" for i in range(len(topic_cols))],
    colors=colors, alpha=0.75,
)

for regime, (y_start, y_end) in CRISIS_REGIMES.items():
    ax.axvspan(y_start - 0.4, y_end + 0.4, alpha=0.1, color="black")
    ax.text((y_start + y_end) / 2, 0.97, regime,
            transform=ax.get_xaxis_transform(), fontsize=7.5, ha="center", va="top")

ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Mean Topic Share", fontsize=12)
ax.set_title("Thematic Evolution: S&P 500 MD&A Topics (2012-2024)", fontsize=13)
ax.legend(loc="upper left", fontsize=7, ncol=3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "topic_trends.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Topic distribution by GICS sector
topic_by_sector = df_topics.groupby("sector")[topic_cols].mean()

fig, ax = plt.subplots(figsize=(12, 6))
topic_by_sector.plot(kind="bar", ax=ax, stacked=True,
                     colormap="tab20", alpha=0.8, width=0.7)
ax.set_xlabel("Sector", fontsize=11)
ax.set_ylabel("Mean Topic Share", fontsize=11)
ax.set_title("Topic Distribution by GICS Sector", fontsize=13)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.legend(loc="upper right", fontsize=7, ncol=2,
          labels=[f"T{i}: {topic_labels.get(i,'')}" for i in range(len(topic_cols))])
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "topic_by_sector.png", dpi=150, bbox_inches="tight")
plt.show()

### Phase 5 Conclusion: LDA Topic Modelling

Coherence-validated LDA revealed the latent thematic structure of the S&P 500 MD&A corpus:

- Optimal Topics: C_v coherence validation identified k = 16 as optimal, with peak coherence of 0.4592. The coherence curve rises from k = 5 (C_v = 0.3751) to a peak at k = 16 achieving the highest C_v score of 0.4592 across all values tested from k=5 to k=20, providing a clear validation signal for the model specification.

- Identifiable Themes: The 16 topics map to interpretable business domains. Topic 2 (gas, risks, production, oil, natural) and Topic 3 (production, gas, oil, natural, prices, conocophillips) both represent Energy sector operations, reflecting the dominance of operational and commodity price language in that sector. Topic 5 (abbvie, products, regulatory, health, revenues) captures pharmaceutical regulatory risk disclosures. Topic 12 (mcdonald, restaurants, markets, foreign) represents McDonald's international franchise operations. Topic 14 (risk, card, credit, loans, liquidity) captures financial services risk management. Topic 13 (stock, internal control, reporting, accounting) represents financial reporting governance disclosures common across all sectors.

- Crisis Sensitivity: Risk and regulatory themes show elevated prevalence during COVID-2020, consistent with Bochkay et al. (2023) who document increased uncertainty language in 10-K filings during economic downturns.

- Sector Variation: Energy and Utilities sectors concentrate in sector-specific topics while Technology and Consumer Discretionary show broader thematic diversity, consistent with greater narrative complexity in those sectors.

- Limitation: LDA assumes a bag-of-words representation and topic independence. BERTopic, using sentence-transformer embeddings, would address these assumptions in future work (Grootendorst, 2022).

# Phase 6: Correlation Analysis - Sentiment vs Stock Returns

### Goal
Test whether MD&A sentiment predicts subsequent stock returns and whether this relationship varies by industry and macroeconomic regime.

### Objectives
- Build lagged dataset linking sentiment in year T to returns in year T+1

- Report Pearson and Spearman correlations for all three sentiment metrics

- Decompose results by GICS sector and macroeconomic regime

- Interpret findings in the context of market efficiency and prior literature

### Methodology
Annual returns are computed as the percentage change from the first to last closing price within each calendar year using adjusted prices from Yahoo Finance. The lagged design (year T sentiment predicts year T+1 returns) is theoretically motivated - if language genuinely predicts performance, the effect should appear in future rather than contemporaneous returns. Both Pearson and Spearman correlations are reported. Spearman is included as it is robust to the outlier returns common in equity data.

In [ ]:
# Load stock returns from cache or retrieve from Yahoo Finance
import yfinance as yf

returns_path = DATA_DIR / "stock_returns.csv"

if returns_path.exists():
    print("Loading stock returns from cache.")
    df_returns = pd.read_csv(returns_path)
else:
    print("Downloading stock returns from Yahoo Finance...")
    tickers = [t for t in df_corpus["ticker"].unique().tolist() if t != "AMZN2"]
    records = []
    for ticker in tqdm(tickers, desc="Fetching returns"):
        try:
            hist = yf.download(ticker, start="1994-01-01", end="2025-01-01",
                               progress=False, auto_adjust=True)
            if hist.empty:
                continue
            if isinstance(hist.columns, pd.MultiIndex):
                hist.columns = hist.columns.get_level_values(0)
            hist.index = pd.to_datetime(hist.index)
            hist["year"] = hist.index.year
            for year, group in hist.groupby("year"):
                if len(group) < 10:
                    continue
                annual_return = (group["Close"].iloc[-1] / group["Close"].iloc[0] - 1) * 100
                records.append({"ticker": ticker, "year": year,
                                 "annual_return": float(annual_return)})
        except Exception as e:
            print(f"  Failed {ticker}: {e}")
        time.sleep(0.1)
    df_returns = pd.DataFrame(records)
    df_returns.to_csv(returns_path, index=False)
    print(f"Saved: {len(df_returns):,} company-year observations")

print(f"Returns dataset: {len(df_returns):,} observations")
print(df_returns.head(10).to_string(index=False))

In [ ]:
# Build lagged and contemporaneous analysis datasets
df_all_sent["year_lag"] = df_all_sent["year"] + 1

# Lagged: sentiment in year T linked to return in year T+1
df_lagged = df_all_sent.merge(
    df_returns,
    left_on  = ["ticker", "year_lag"],
    right_on = ["ticker", "year"],
    suffixes = ("_sent", "_ret"),
).rename(columns={"year_sent": "sent_year", "year_ret": "ret_year"})

# Contemporaneous: sentiment in year T vs return in same year T
df_same = df_all_sent.merge(df_returns, on=["ticker", "year"])

df_lagged.to_csv(DATA_DIR / "master_analysis.csv", index=False)
print(f"Lagged analysis pairs:           {len(df_lagged):,}")
print(f"Contemporaneous analysis pairs:  {len(df_same):,}")

In [ ]:
# Correlation tables: lagged and contemporaneous
sentiment_metrics = {}
for col, name in [
    ("vader_compound",   "VADER Compound"),
    ("lm_net_sentiment", "LM Net Sentiment"),
    ("finbert_net",      "FinBERT Net"),
]:
    if col in df_lagged.columns:
        sentiment_metrics[col] = name

def correlation_table(df, return_col, label):
    print(f"{'='*70}")
    print(f"  {label}")
    print(f"{'='*70}")
    print(f"  {'Metric':<25} {'Pearson r':>10} {'p-value':>12} {'Spearman r':>12} {'n':>7}")
    print(f"  {'-'*67}")
    for col, name in sentiment_metrics.items():
        valid = df[[col, return_col]].dropna()
        if len(valid) < 5:
            continue
        pr, pp = stats.pearsonr( valid[col], valid[return_col])
        sr, sp = stats.spearmanr(valid[col], valid[return_col])
        sig = "**" if pp < 0.05 else ("*" if pp < 0.10 else "")
        print(f"  {name:<25} {pr:>+10.3f} {pp:>12.4f} {sr:>+11.3f} {len(valid):>6}  {sig}")
    print(f"  {'-'*67}")
    print("  ** p<0.05  * p<0.10")
    print()

correlation_table(df_lagged, "annual_return", "LAGGED: Sentiment Year T predicts Return Year T+1")
correlation_table(df_same,   "annual_return", "CONTEMPORANEOUS: Sentiment Year T vs Return Year T")

In [ ]:
# Sector-level correlations (FinBERT net vs lagged returns)
print("Sector-level Pearson correlations (FinBERT Net vs Lagged Return):")
print("=" * 55)

if "finbert_net" in df_lagged.columns:
    sector_corrs = []
    for sector, grp in df_lagged.groupby("sector"):
        valid = grp[["finbert_net", "annual_return"]].dropna()
        if len(valid) < 5:
            continue
        r, p = stats.pearsonr(valid["finbert_net"], valid["annual_return"])
        sector_corrs.append({"Sector": sector, "r": round(r, 3),
                             "p": round(p, 4), "n": len(valid)})
    sector_df = pd.DataFrame(sector_corrs).sort_values("r", ascending=False)
    print(sector_df.to_string(index=False))

In [ ]:
# Regime-level correlations
print("Regime-level Pearson correlations (FinBERT Net vs Lagged Return):")
print("=" * 55)

if "finbert_net" in df_lagged.columns:
    def get_regime(year):
        for regime, (y_start, y_end) in CRISIS_REGIMES.items():
            if y_start <= year <= y_end:
                return regime
        return "Stable"

    df_lagged["regime"] = df_lagged["sent_year"].apply(get_regime)
    for regime, grp in df_lagged.groupby("regime"):
        valid = grp[["finbert_net", "annual_return"]].dropna()
        if len(valid) < 5:
            continue
        r, p = stats.pearsonr(valid["finbert_net"], valid["annual_return"])
        print(f"  {regime:<20} r={r:+.3f}  p={p:.4f}  n={len(valid)}")

In [ ]:
# Scatter plot: FinBERT net sentiment vs lagged returns, coloured by sector
if "finbert_net" in df_lagged.columns:
    valid = df_lagged[["finbert_net", "annual_return", "sector"]].dropna()
    fig, ax = plt.subplots(figsize=(10, 6))
    sectors = valid["sector"].unique()
    palette = sns.color_palette("tab10", len(sectors))
    for sector in sectors:
        sub = valid[valid["sector"] == sector]
        ax.scatter(sub["finbert_net"], sub["annual_return"],
                   c=[palette[list(sectors).index(sector)]], s=20, alpha=0.5, label=sector)
    m, b, r, p, _ = stats.linregress(valid["finbert_net"], valid["annual_return"])
    xs = np.linspace(valid["finbert_net"].min(), valid["finbert_net"].max(), 100)
    ax.plot(xs, m*xs+b, "k--", linewidth=2, label=f"Overall: r={r:.3f}, p={p:.4f}")
    ax.set_xlabel("FinBERT Net Sentiment (MD&A Year T)", fontsize=12)
    ax.set_ylabel("Annual Stock Return % (Year T+1)", fontsize=12)
    ax.set_title("FinBERT Sentiment vs Subsequent Stock Returns (2012-2024)", fontsize=12)
    ax.legend(fontsize=7, ncol=2, loc="upper left")
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "finbert_vs_returns.png", dpi=150, bbox_inches="tight")
    plt.show()

### Phase 6 Conclusion: Sentiment-Returns Correlation

The correlation analysis across 350 lagged company-year pairs directly answers the research question:

- Overall Lagged Signal: None of the three sentiment metrics show a statistically significant lagged relationship with next-year stock returns. VADER Compound: Pearson r = -0.024 (p = 0.649). LM Net Sentiment: r = +0.033 (p = 0.544). FinBERT Net: r = +0.026 (p = 0.632). All correlations are near zero and far below the minimum detectable effect of r = 0.150.

- Contemporaneous Comparison: Contemporaneous correlations are marginally stronger. VADER: r = +0.083 (p = 0.089, borderline significant at 10% level). LM: r = +0.057 (p = 0.244). FinBERT: r = +0.054 (p = 0.269). The pattern of contemporaneous correlations being consistently stronger than lagged correlations suggests MD&A language primarily reflects performance conditions already known at time of writing, rather than predicting future performance.

- Sector Decomposition: At the sector level, Financials shows the strongest lagged correlation (r = +0.365, n = 14) and Communication the most negative (r = -0.467, n = 15). However, all sector-level results have small n values and none reach p < 0.05, preventing sector-level inference.

- Regime Analysis: Correlations remain near zero across all macroeconomic regimes. COVID: r = +0.169 (p = 0.222, n = 54). Inflation: r = +0.009 (p = 0.923, n = 125). Stable: r = +0.017 (p = 0.823, n = 171). The absence of meaningful signal is consistent rather than regime-dependent.

- Interpretation: Near-zero lagged correlations are themselves a substantive finding consistent with the efficient market hypothesis. If MD&A language systematically predicted future returns, rational investors would already incorporate that signal into current prices. The modest contemporaneous correlation for VADER (r = +0.083) is consistent with Kothari et al. (2009), who find that forward-looking 10-K disclosures are systematically more optimistic than realised outcomes, meaning contemporaneous language reflects known performance rather than forecasting unknown performance.

# Phase 7: Research Synthesis, Limitations and Future Directions

### Research Question Revisited
Does the linguistic sentiment of CEO annual narratives in S&P 500 10-K filings predict subsequent stock returns, and does this relationship vary across industries and macroeconomic regimes?

### Answer
The empirical evidence from this corpus does not support a significant lagged predictive relationship between MD&A sentiment and subsequent annual stock returns at any conventional significance threshold. All three sentiment methods (VADER, LM, FinBERT) produce near-zero lagged correlations across all sectors and all macroeconomic regimes. The result is consistent with the efficient market hypothesis: information embedded in publicly disclosed MD&A text is rapidly incorporated into prices before the annual measurement window. The modest contemporaneous correlation for VADER (r = +0.083, borderline significant) suggests language co-moves with known performance rather than predicting unknown future performance.

In [ ]:
# Final synthesis table summarising all methods and key findings
synthesis = pd.DataFrame({
    "Phase":  ["3 - VADER", "3 - LM Dictionary", "4 - FinBERT", "5 - LDA"],
    "Type":   ["Lexicon (General)", "Lexicon (Finance)",
               "Transfer Learning", "Unsupervised"],
    "Validation": [
        "Comparison with FinBERT (r=0.481)",
        "Loughran and McDonald (2011)",
        "Pearson r vs VADER (r=0.481)",
        "C_v Coherence Score (k=16, C_v=0.459)",
    ],
    "Key Finding": [
        "Broadly positive; contemporaneous r=+0.083 (p=0.089)",
        "Negative ratio consistently higher; LM net negative across corpus",
        "Lower positivity than VADER; lagged r=+0.026 (p=0.632)",
        "16 topics; sector-specific themes; COVID risk shift visible",
    ],
})
print("Research Synthesis:")
print(synthesis.to_string(index=False))

In [ ]:
# Statistical power assessment
n_lagged = len(df_lagged.dropna(subset=["finbert_net", "annual_return"]))
r_min    = 2.8 / (n_lagged ** 0.5)

print("Statistical Power Assessment")
print("=" * 50)
print(f"  Lagged analysis observations:  {n_lagged:,}")
print(f"  Min detectable r (80% power):  ~{r_min:.3f}")
print(f"  Published benchmark (LM 2011): r = 0.10-0.15")
print()
print("  This corpus is 7x larger than PS1's effective sample.")
print("  The near-zero lagged correlations fall below the")
print("  detection threshold, making the null result genuine")
print("  rather than a consequence of insufficient power.")

### Limitations

1. MD&A as CEO Voice Proxy: The MD&A section is drafted by executive teams with legal and communications input and is not directly attributable to the CEO. Future work using earnings call transcripts would provide a more conversational and directly attributable narrative source at quarterly frequency.

2. Corpus Temporal Scope: The effective corpus spans 2012 to 2024. The originally planned 1995-2024 window would have included the Dotcom bust and GFC as additional crisis regimes, potentially revealing regime-dependent signal not detectable in the current window.

3. Survivorship Bias: The company list reflects current S&P 500 constituents. Companies that failed, merged, or were delisted are underrepresented. This biases the corpus toward surviving firms and may cause sentiment scores to be systematically higher than a complete historical sample would produce.

4. Endogeneity: 10-K filings are published in February or March after management knows the full-year results. Language may reflect known performance rather than genuine forward-looking views, making it difficult to separate predictive from retrospective language without intra-year data.

5. FinBERT Training Domain: FinBERT was fine-tuned on financial news headlines rather than long-form regulatory filings. A model trained specifically on the EDGAR corpus would better capture the formal, hedged language of MD&A sections.

6. Annual Return Window: Annual returns may be too coarse to detect sentiment effects that are priced within days of the filing date. Event-study returns computed around the 10-K filing date would provide a sharper and more theoretically motivated test.

### Future Directions

1. Event-Study Returns: Using three-day cumulative abnormal returns around the 10-K filing date would isolate the market reaction to new MD&A information and avoid the endogeneity concern about retrospective language.

2. SEC-Specific Language Model: Pre-training a BERT model on the full EDGAR 10-K archive would address FinBERT's training domain mismatch and likely improve both sentiment classification accuracy and downstream correlation strength.

3. Earnings Call Transcripts: Quarterly earnings call transcripts are more conversational, directly CEO-attributed, and available at higher frequency. Combining annual 10-K MD&A with quarterly call sentiment would test whether the predictive signal is stronger in less legally constrained disclosures.

4. Extended Historical Corpus: Extending the pipeline back to 1995 using the full EDGAR historical archive would include the Dotcom bust and GFC as additional crisis regimes, enabling a more comprehensive test of the regime-dependence hypothesis.

5. Joint Model with Financial Ratios: Testing whether sentiment adds incremental predictive power beyond standard accounting ratios (leverage, profitability, growth) in a joint regression would establish whether linguistic signal carries information beyond quantitative fundamentals.